# 🪸 Coral Bleaching Prediction — CNN + GRU
**Model:** CNN extracts spatial features, GRU learns temporal trends (faster & simpler than LSTM)

**Difference from CNN+LSTM:** GRU uses 2 gates instead of 3 → fewer parameters, faster training, often similar accuracy

**Input:** 7 consecutive days × (4, 224, 224) stacked images

**Output:** BAA label 7 days ahead (0–4)

**Train:** 2018–2023 | **Test:** 2024–2025

## Step 1: Upload Dataset

In [ ]:
from google.colab import files
import zipfile, os

print("Click 'Choose Files' and select 3_dataset.zip ...")
uploaded = files.upload()

with zipfile.ZipFile('3_dataset.zip', 'r') as z:
    z.extractall('.')
print('Unzipped!')

DATASET_DIR = '/content/3_dataset'
SAVE_DIR    = '/content/4_models'
RESULTS_DIR = '/content/5_results'

os.makedirs(SAVE_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Dataset ready!')
print('Train files:', len(os.listdir(f'{DATASET_DIR}/train')))
print('Test files: ', len(os.listdir(f'{DATASET_DIR}/test')))

## Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## Step 3: Dataset — 7-Day Forecast

In [ ]:
FORECAST = 7

class CoralSequenceDataset(Dataset):
    """
    Loads sequences of 7 consecutive days.
    Input shape:  (7, 4, 224, 224)
    Label:        BAA value 7 days ahead
    """
    def __init__(self, dataset_dir, labels_csv, seq_len=7):
        self.dataset_dir = dataset_dir
        self.seq_len     = seq_len

        df = pd.read_csv(labels_csv)
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)

        valid = []
        for _, row in df.iterrows():
            d    = row['date']
            path = f"{dataset_dir}/stacked_{d.year}_{d.month:02d}_{d.day:02d}.npy"
            if os.path.exists(path):
                valid.append({'date': d, 'label': int(row['label']), 'path': path})

        self.records = valid
        print(f'  Valid samples: {len(self.records)}')
        print(f'  Sequences available: {len(self)}')

    def __len__(self):
        return len(self.records) - self.seq_len - FORECAST + 1

    def __getitem__(self, idx):
        frames = []
        for i in range(self.seq_len):
            arr = np.load(self.records[idx + i]['path'])
            frames.append(arr)
        sequence = np.stack(frames, axis=0)
        label    = self.records[idx + self.seq_len - 1 + FORECAST]['label']
        return torch.tensor(sequence, dtype=torch.float32), torch.tensor(label, dtype=torch.long)


SEQ_LEN    = 7
BATCH_SIZE = 16

print('Loading train dataset...')
train_dataset = CoralSequenceDataset(
    f'{DATASET_DIR}/train',
    f'{DATASET_DIR}/train_labels.csv',
    seq_len=SEQ_LEN
)

print('\nLoading test dataset...')
test_dataset = CoralSequenceDataset(
    f'{DATASET_DIR}/test',
    f'{DATASET_DIR}/test_labels.csv',
    seq_len=SEQ_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

x, y = next(iter(train_loader))
print(f'\nBatch input shape: {x.shape}  → (batch, seq, channels, H, W)')
print(f'Batch label shape: {y.shape}')

## Step 4: Class Weights

In [ ]:
all_labels = [train_dataset.records[i + SEQ_LEN - 1 + FORECAST]['label']
              for i in range(len(train_dataset))]

classes = np.array([0, 1, 2, 3, 4])
weights = compute_class_weight('balanced', classes=classes, y=all_labels)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print('Class weights:')
for i, w in enumerate(weights):
    print(f'  Class {i}: {w:.3f}')

## Step 5: CNN + GRU Model
**Key difference from CNN+LSTM:**
- LSTM has 3 gates: input, forget, output
- GRU has 2 gates: reset, update
- GRU is faster to train and has fewer parameters
- Often achieves similar or better accuracy than LSTM

In [ ]:
class CNNFeatureExtractor(nn.Module):
    """CNN to extract spatial features — same as CNN+LSTM version."""
    def __init__(self, in_channels=4, feature_dim=256):
        super().__init__()
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Linear(256 * 4 * 4, feature_dim)

    def forward(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


class CoralCNNGRU(nn.Module):
    """
    CNN + GRU model.
    - CNN extracts spatial features from each day (same as CNN+LSTM)
    - GRU learns temporal trends (replaces LSTM)
    - GRU: fewer parameters, faster, often similar accuracy
    """
    def __init__(self, feature_dim=256, hidden_dim=128, num_layers=2, num_classes=5):
        super().__init__()
        self.cnn = CNNFeatureExtractor(in_channels=4, feature_dim=feature_dim)

        # ── KEY DIFFERENCE: GRU instead of LSTM ──────────────
        self.gru = nn.GRU(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )
        # ─────────────────────────────────────────────────────

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x: (batch, seq_len, 4, 224, 224)
        batch, seq_len, C, H, W = x.shape

        # Apply CNN to each day
        cnn_out = []
        for t in range(seq_len):
            feat = self.cnn(x[:, t, :, :, :])
            cnn_out.append(feat)

        # Stack: (batch, seq_len, feature_dim)
        cnn_out = torch.stack(cnn_out, dim=1)

        # ── GRU instead of LSTM ───────────────────────────────
        gru_out, _ = self.gru(cnn_out)    # (batch, seq_len, hidden_dim)
        last_out   = gru_out[:, -1, :]    # take last timestep
        # ─────────────────────────────────────────────────────

        return self.classifier(last_out)


model = CoralCNNGRU(
    feature_dim=256,
    hidden_dim=128,
    num_layers=2,
    num_classes=5
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')
print('\nNote: CNN+LSTM had more parameters due to 3 gates vs GRU\'s 2 gates')

## Step 6: Early Stopping Class

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience    = patience
        self.min_delta   = min_delta
        self.counter     = 0
        self.best_acc    = 0.0
        self.should_stop = False

    def check(self, val_acc):
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc = val_acc
            self.counter  = 0
        else:
            self.counter += 1
            print(f'  No improvement {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.should_stop = True

## Step 7: Training Setup

In [ ]:
EPOCHS = 30
LR     = 1e-3

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
print(f'Training CNN+GRU for {EPOCHS} epochs on {device}')

## Step 8: Train

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == y).sum().item()
        total      += y.size(0)
    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out  = model(X)
            loss = criterion(out, y)
            total_loss += loss.item()
            correct    += (out.argmax(1) == y).sum().item()
            total      += y.size(0)
    return total_loss / len(loader), correct / total


best_val_acc   = 0.0
early_stopping = EarlyStopping(patience=5)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss,   val_acc   = eval_epoch(model,  test_loader,  criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'{SAVE_DIR}/cnn_gru_best.pth')

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}'
          + (' <- best saved' if val_acc == best_val_acc else ''))

    early_stopping.check(val_acc)
    if early_stopping.should_stop:
        print(f'\n Early stopping triggered at epoch {epoch}!')
        break

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

## Step 9: Plot Training History

In [ ]:
actual_epochs = len(history['train_loss'])
print(f'Epochs trained: {actual_epochs}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', color='blue')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='orange')
axes[0].set_title('CNN+GRU — Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train Acc', color='blue')
axes[1].plot(history['val_acc'],   label='Val Acc',   color='orange')
axes[1].set_title('CNN+GRU — Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/cnn_gru_training.png', dpi=150)
plt.show()
print('Saved training plot!')

## Step 10: Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load(f'{SAVE_DIR}/cnn_gru_best.pth'))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for X, y in test_loader:
        X     = X.to(device)
        out   = model(X)
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average='weighted')

print('='*50)
print('CNN+GRU — TEST RESULTS')
print('='*50)
print(f'Accuracy:          {acc:.4f}')
print(f'Weighted F1 Score: {f1:.4f}')
print()
print(classification_report(
    all_labels, all_preds,
    target_names=['No Stress', 'Watch', 'Warning', 'Alert L1', 'Alert L2']
))

## Step 11: Confusion Matrix

In [ ]:
cm     = confusion_matrix(all_labels, all_preds)
labels = ['No Stress', 'Watch', 'Warning', 'Alert L1', 'Alert L2']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('CNN+GRU — Confusion Matrix (Test Set 2024–2025)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/cnn_gru_confusion.png', dpi=150)
plt.show()
print('Saved confusion matrix!')

## Step 12: Compare GRU vs LSTM

In [ ]:
print('='*50)
print('CNN+GRU vs CNN+LSTM COMPARISON')
print('='*50)
print(f'CNN+GRU  Accuracy: {acc:.4f}')
print(f'CNN+LSTM Accuracy: 0.8375  (from previous experiment)')
print()
if acc > 0.8375:
    print('GRU WINS! Simpler model, better accuracy')
elif acc > 0.8300:
    print('GRU is competitive — similar accuracy with fewer parameters')
else:
    print('LSTM wins — more complex gating helps for this dataset')

## Step 13: Save & Download Results

In [ ]:
import json

results = {
    'model':         'CNN+GRU',
    'accuracy':      round(float(acc), 4),
    'f1_weighted':   round(float(f1), 4),
    'best_val_acc':  round(float(best_val_acc), 4),
    'epochs':        actual_epochs,
    'forecast_days': FORECAST,
    'seq_len':       SEQ_LEN,
    'note':          'GRU replaces LSTM — 2 gates instead of 3'
}

with open(f'{RESULTS_DIR}/cnn_gru_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

In [ ]:
from google.colab import files
files.download(f'{SAVE_DIR}/cnn_gru_best.pth')

In [ ]:
files.download(f'{RESULTS_DIR}/cnn_gru_results.json')

In [ ]:
files.download(f'{RESULTS_DIR}/cnn_gru_training.png')

In [ ]:
files.download(f'{RESULTS_DIR}/cnn_gru_confusion.png')
print('All downloaded!')